# Generate Outline - Step by Step Testing
## Flow: User selects files → clicks "Generate Outline" → No context prompt

This notebook traces the **exact code path** when a user:
1. Selects files from the frontend
2. Clicks "Generate Outline" button
3. Does NOT provide any context prompt

### Execution Order:
1. **Setup** - Imports & Environment
2. **Input Simulation** - Mimic the frontend request (OutlineRequest)
3. **Validation** - File type checks
4. **Service Layer** - OutlineService.generate_outline()
5. **Orchestrator Pipeline** - search_outline_pipeline()
6. **Search Engine** - OrchestratorAgent.process_search_request()
   - Build OData filter
   - Step 1: Topic detection (empty → fetch all)
   - Process raw results (TOC filter, images, durations, sorting)
   - Steps 2-9 (all skipped since no prompt)
   - Sequencing (page sort or LLM logical)
   - Save results (inject intro, knowledge checks, summary, assessment)
7. **Result Mapping** - Map JSON to outline items

## Step 1: Setup - Imports & Environment Variables
Load all dependencies and configure Azure clients (same as production code).

In [ ]:
import os
import sys
import json
import re
import time
import uuid
import logging
import pandas as pd
from typing import List, Dict, Any, Optional
from datetime import datetime

from dotenv import load_dotenv
load_dotenv()

# Azure SDK Imports
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizableTextQuery
from openai import AzureOpenAI

# Configure logging to see pipeline steps
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("All imports loaded successfully.")

In [ ]:
# Verify environment variables are loaded
env_vars = {
    "AZURE_SEARCH_ENDPOINT": os.getenv("AZURE_SEARCH_ENDPOINT"),
    "AZURE_SEARCH_API_KEY": os.getenv("AZURE_SEARCH_API_KEY"),
    "AZURE_SEARCH_INDEX_NAME": os.getenv("AZURE_SEARCH_INDEX_NAME"),
    "AZURE_OPENAI_ENDPOINT": os.getenv("AZURE_OPENAI_ENDPOINT"),
    "AZURE_OPENAI_API_KEY": os.getenv("AZURE_OPENAI_API_KEY"),
    "AZURE_OPENAI_CHAT_DEPLOYMENT_NAME": os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME"),
}

for key, val in env_vars.items():
    status = "SET" if val else "MISSING"
    print(f"  {key}: {status}")

SEARCH_ENDPOINT = env_vars["AZURE_SEARCH_ENDPOINT"]
SEARCH_API_KEY = env_vars["AZURE_SEARCH_API_KEY"]
SEARCH_INDEX_NAME = env_vars["AZURE_SEARCH_INDEX_NAME"]
AZURE_OPENAI_ENDPOINT = env_vars["AZURE_OPENAI_ENDPOINT"]
AZURE_OPENAI_API_KEY = env_vars["AZURE_OPENAI_API_KEY"]
AZURE_OPENAI_CHAT_DEPLOYMENT_NAME = env_vars["AZURE_OPENAI_CHAT_DEPLOYMENT_NAME"]
OPENAI_CHAT_API_VERSION = "2024-02-15-preview"
GLOBAL_SEED = 42

## Step 2: Simulate Frontend Request (OutlineRequest)
This is what the frontend sends when user selects files and clicks "Generate Outline" with **no context prompt**.

**Change these values to match your test data:**

In [ ]:
# ============================================================
# CHANGE THESE VALUES TO MATCH YOUR TEST DATA
# ============================================================
CLIENT = "Roshan_Jan"           # Your client name
PROJECT = "MyCC"                # Your project name
MODULE = "Module1"              # Your module name
FILES = ["Day 4 - Networking Connections.pptx"]  # Selected files
CONTEXT_PROMPT = None           # No context prompt (this is the scenario we're testing)

# This is the equivalent of OutlineRequest from the frontend
input_data = {
    "client": CLIENT,
    "project": PROJECT,
    "module": MODULE,
    "files": FILES,
    "context_prompt": CONTEXT_PROMPT  # None = no prompt given by user
}

print("Frontend Request (OutlineRequest):")
print(json.dumps(input_data, indent=2, default=str))

## Step 3: Validation - File Type Check
**From:** `outline_routes.py` → `generate_outline()` (lines 177-190)

Before starting background job, the route validates file extensions.

In [ ]:
# --- From outline_routes.py lines 71-88 ---
ALLOWED_EXTENSIONS = {
    '.doc', '.docx', '.pdf',
    '.xls', '.xlsx', '.csv',
    '.ppt', '.pptx',
    '.mp3', '.wav', '.m4a', '.aac', '.flac', '.ogg',
    '.mp4', '.mov', '.avi', '.mkv', '.wmv', '.flv'
}

def is_supported_file(filename: str) -> bool:
    ext = os.path.splitext(filename.lower())[1]
    return ext in ALLOWED_EXTENSIONS

# --- Validation (from outline_routes.py lines 177-190) ---
unsupported_files = [f for f in FILES if not is_supported_file(f)]

if unsupported_files:
    print(f"BLOCKED: Unsupported files: {unsupported_files}")
else:
    print("All files passed validation.")
    for f in FILES:
        ext = os.path.splitext(f.lower())[1]
        print(f"  {f} -> extension '{ext}' is supported")

## Step 4: Gibberish Check (Skipped)
**From:** `outline_routes.py` → `generate_outline()` (lines 148-156)

Since `context_prompt` is `None`, the gibberish check is **skipped entirely**.
```python
if input_data.context_prompt and is_gibberish(input_data.context_prompt):
    # This block is NEVER entered when context_prompt is None
```

In [ ]:
# --- From outline_routes.py line 149 ---
# Gibberish check is short-circuited because context_prompt is None (falsy)
if CONTEXT_PROMPT and False:  # is_gibberish(CONTEXT_PROMPT) would be called here
    print("BLOCKED: Gibberish detected")
else:
    print("Gibberish check SKIPPED (context_prompt is None)")
    print("  -> Condition: `if input_data.context_prompt` evaluates to False when None")

## Step 5: Build selection_data
**From:** `outline_service.py` → `OutlineService.generate_outline()` (lines 64-71)

The service layer extracts the selection data from the request and passes it to the orchestrator pipeline.

In [ ]:
# --- From outline_service.py lines 64-71 ---
# OutlineService.generate_outline() builds this dict from OutlineRequest
selection_data = {
    "client": input_data["client"],
    "project": input_data["project"],
    "module": input_data["module"],
    "files": input_data["files"],
    "context_prompt": input_data["context_prompt"]  # None in our case
}

print("selection_data passed to search_outline_pipeline():")
print(json.dumps(selection_data, indent=2, default=str))

## Step 6: Initialize Azure Clients (BaseSearchAgent)
**From:** `search_service.py` → `BaseSearchAgent.__init__()` (lines 45-69)

Creates the Azure Search Client and Azure OpenAI client.

In [ ]:
# --- From search_service.py lines 59-69 (BaseSearchAgent._initialize_clients) ---
try:
    search_client = SearchClient(
        SEARCH_ENDPOINT, 
        SEARCH_INDEX_NAME, 
        AzureKeyCredential(SEARCH_API_KEY)
    )
    chat_client = AzureOpenAI(
        api_key=AZURE_OPENAI_API_KEY,
        api_version=OPENAI_CHAT_API_VERSION,
        azure_endpoint=AZURE_OPENAI_ENDPOINT
    )
    deployment_name = AZURE_OPENAI_CHAT_DEPLOYMENT_NAME
    
    # Field mapping used throughout the pipeline
    json_field_mapping = {
        "File": "File", "Source Page": "Source Page", "Chapter": "Chapter",
        "Topic": "Topic", "Subtopic": "Subtopic", "Full Page Content": "Full Page Content",
        "Content_without_images": "Content_without_images",
        "Durations (Mins)": "Durations (Mins)"
    }
    
    print("Clients Connected Successfully.")
    print(f"  Search Index: {SEARCH_INDEX_NAME}")
    print(f"  OpenAI Deployment: {deployment_name}")
except Exception as e:
    print(f"Connection Failed: {e}")

## Step 7: Build OData Filter
**From:** `search_service.py` → `BaseSearchAgent.build_odata_filter()` (lines 166-174)

Constructs the Azure Search filter from client, project, and file names.

In [ ]:
# --- From search_service.py lines 166-174 (BaseSearchAgent.build_odata_filter) ---
def build_odata_filter(client=None, project=None, files=None):
    parts = []
    if client: parts.append(f"Client eq '{client}'")
    if project: parts.append(f"Project eq '{project}'")
    if files:
        f_list = [files] if isinstance(files, str) else files
        f_parts = [f"File eq '{f}'" for f in f_list]
        parts.append(f"({' or '.join(f_parts)})")
    return " and ".join(parts) if parts else None

# Build filter for our test data
active_filter = build_odata_filter(
    client=selection_data["client"],
    project=selection_data["project"],
    files=selection_data["files"]
)

print("OData Filter:")
print(f"  {active_filter}")

## Step 8: STEP 1 - Topic Detection & Decision Matrix
**From:** `search_service.py` → `OrchestratorAgent.process_search_request()` (lines 1106-1127)

Since `context_prompt` is None (empty string after `.strip()`):
- `extract_smart_topics("")` returns `[]` (no topics)
- `extract_page_constraints("")` returns `None` (no page range)
- **Decision:** Falls into `elif files:` branch → executes `fetch_all`

In [ ]:
# --- From search_service.py lines 1088-1127 (OrchestratorAgent.process_search_request) ---

# Process the prompt (None becomes empty string)
prompt = (selection_data.get("context_prompt") or "").strip()
files = selection_data.get("files", [])
manual_topics = []  # No topics in our request

print(f"Prompt after strip: '{prompt}'")
print(f"Files: {files}")
print(f"Manual topics: {manual_topics}")

# --- STEP 1a: extract_smart_topics (from search_service.py line 281-282) ---
# When prompt is empty, returns [] immediately
def extract_smart_topics(prompt_text):
    if not prompt_text: return []
    # ... (LLM call would happen here if prompt was not empty)
    return []

detected_topics = extract_smart_topics(prompt)
all_topics = list(set(manual_topics + detected_topics))
print(f"\nDetected topics: {detected_topics}")
print(f"All topics: {all_topics}")

# --- STEP 1b: extract_page_constraints (from search_service.py line 332-341) ---
def extract_page_constraints(prompt_text):
    pattern = r'(?:page|pg|slide)[s]?\s*(\d+)\s*(?:to|-|through)\s*(\d+)'
    match = re.search(pattern, prompt_text, re.IGNORECASE)
    if match:
        return (int(match.group(1)), int(match.group(2)))
    return None

page_range = extract_page_constraints(prompt)
print(f"Page range: {page_range}")

# --- STEP 1c: Decision Matrix ---
print(f"\n--- Decision Matrix ---")
if all_topics:
    print("  -> Branch: TOPIC DETECTED (search_detected_topics)")
elif files:
    print("  -> Branch: NO SEMANTIC TOPIC + FILES PRESENT -> execute_fetch_all")
    print("     This fetches ALL content for the selected files from Azure Search")
else:
    print("  -> Branch: FALLBACK (hybrid search)")

## Step 9: Execute Fetch All (Azure Search)
**From:** `search_service.py` → `BaseSearchAgent.execute_fetch_all()` (lines 228-236)

Fetches ALL chunks from Azure AI Search for the selected files using `search_text="*"` (wildcard).

In [ ]:
# --- From search_service.py lines 228-236 (BaseSearchAgent.execute_fetch_all) ---
print(f"Executing fetch_all with filter: {active_filter}")
print(f"Search text: '*' (wildcard = fetch everything)")
print(f"Top: 2000 (max chunks to return)")
print()

raw_results = list(search_client.search(
    search_text="*",
    select=["id", "content", "chapter", "topic", "subtopic", "source_page_range", "chunk_number", "File", "images", "table_images"],
    filter=active_filter,
    top=2000
))

print(f"Raw results returned: {len(raw_results)} chunks")
if raw_results:
    print(f"\nSample raw result (first chunk):")
    sample = {k: v for k, v in dict(raw_results[0]).items() if k != 'content' and k != 'images' and k != 'table_images'}
    print(json.dumps(sample, indent=2, default=str))
    content_preview = str(dict(raw_results[0]).get('content', ''))[:200]
    print(f"\nContent preview: {content_preview}...")

## Step 10: Process Raw Results (TOC Filter, Images, Duration, Sorting)
**From:** `search_service.py` → `BaseSearchAgent.process_raw_results()` (lines 103-163)

This is a critical step that:
1. Filters out Table of Contents / Index pages
2. Calculates word-based duration for each chunk
3. Parses and replaces image placeholders with base64 data
4. Renames columns to output format
5. Sorts by File → Page Number → Chunk Number

In [ ]:
# --- From search_service.py lines 103-163 (BaseSearchAgent.process_raw_results) ---

def _is_toc_or_index_row(row: pd.Series) -> bool:
    """Filter out Table of Contents and Index pages"""
    chapter = str(row.get('chapter', '')).lower()
    topic = str(row.get('topic', '')).lower()
    content = str(row.get('content', '')).lower()
    bad_keywords = ["table of contents", "index", "list of tables", "list of figures"]
    if any(k == chapter for k in bad_keywords) or any(k == topic for k in bad_keywords):
        return True
    if content.count('....') > 5:  # Dotted TOC lines
        return True
    return False

def _format_duration(text: str) -> str:
    """Calculate reading duration based on word count (2.5 words/sec)"""
    if not text: return "0 min 0 sec"
    word_count = len(text.split())
    seconds = word_count / 2.5
    return f"{int(seconds // 60)} min {int(seconds % 60)} sec"

def process_raw_results(raw_hits: list) -> list:
    if not raw_hits: return []
    
    # Convert to DataFrame
    df_raw = pd.DataFrame(raw_hits)
    print(f"  Raw DataFrame shape: {df_raw.shape}")
    
    # Step 10a: Filter TOC/Index rows
    mask = df_raw.apply(_is_toc_or_index_row, axis=1)
    df_filtered = df_raw[~mask].copy()
    toc_removed = mask.sum()
    print(f"  TOC/Index rows removed: {toc_removed}")
    print(f"  After filtering: {len(df_filtered)} rows")
    
    if df_filtered.empty: return []
    
    processed_rows = []
    for _, row in df_filtered.iterrows():
        row_dict = row.to_dict()
        content_text = row_dict.get('content', '')
        chunk_id = row_dict.get('id', 'unknown')
        
        # Step 10b: Duration Calculation
        word_count = len(content_text.split())
        row_dict['duration_seconds'] = (word_count / 2.5) if word_count > 0 else 0
        
        # Step 10c: Image Parsing
        image_data_map = {}
        images_json = row_dict.get('images')
        if images_json:
            try:
                images_list = json.loads(images_json) if isinstance(images_json, str) else images_json
                for img_obj in images_list:
                    iid = str(img_obj.get('id') or img_obj.get('image_id') or '').strip().lower()
                    b64 = (img_obj.get('data') or img_obj.get('base64') or '').strip()
                    if iid and b64: image_data_map[iid] = b64
            except: pass
        
        # Step 10d: Image Replacement in content
        def replace_regular_image(match):
            image_id = match.group(1).strip().lower()
            base64_data = image_data_map.get(image_id)
            return f"![Image {image_id}](data:image/png;base64,{base64_data})" if base64_data else f"[MISSING_IMAGE:{image_id}]"
        
        content_text = re.sub(r'\[IMAGE:\s*([^\]]+?)\]', replace_regular_image, content_text, flags=re.IGNORECASE)
        row_dict['content'] = content_text
        row_dict['Full Page Content'] = content_text
        row_dict['Content_without_images'] = re.sub(r'!\[.*?\]\(data:image/.*?;base64,.*?\)', '[IMAGE_REMOVED]', content_text)
        row_dict['Durations (Mins)'] = _format_duration(content_text)
        processed_rows.append(row_dict)
    
    # Step 10e: Sorting & Renaming
    df_processed = pd.DataFrame(processed_rows)
    df_renamed = df_processed.rename(columns={
        'chapter': 'Chapter', 'topic': 'Topic', 'subtopic': 'Subtopic',
        'source_page_range': 'Source Page', 'File': 'File'
    })
    df_renamed['start_page'] = pd.to_numeric(
        df_renamed['Source Page'].astype(str).str.extract(r'(\d+)')[0], errors='coerce'
    ).fillna(0)
    if 'chunk_number' not in df_renamed.columns:
        df_renamed['chunk_number'] = 0
    
    df_sorted = df_renamed.sort_values(
        by=["File", "start_page", "chunk_number"], ascending=[True, True, True]
    )
    
    final_results = []
    for _, row in df_sorted.iterrows():
        row_dict = row.to_dict()
        processed_row = {'id': row_dict.get('id'), 'duration_seconds': row_dict.get('duration_seconds')}
        for json_key, df_column_name in json_field_mapping.items():
            processed_row[json_key] = row_dict.get(df_column_name)
        final_results.append(processed_row)
    
    return final_results

# Execute processing
current_pool = process_raw_results(raw_results)
print(f"\nProcessed results: {len(current_pool)} chunks")
if current_pool:
    print(f"\nSample processed chunk:")
    sample = {k: v for k, v in current_pool[0].items() if k != 'Full Page Content' and k != 'Content_without_images'}
    print(json.dumps(sample, indent=2, default=str))

## Step 11: Pipeline Steps 2-7 (ALL SKIPPED - No Prompt)
**From:** `search_service.py` → `OrchestratorAgent.process_search_request()` (lines 1146-1186)

Since `prompt` is empty (`""`), **all keyword-triggered steps are skipped**:
- Step 2 (Inclusion/Exclusion): needs "include", "exclude", "only", etc.
- Step 3 (Audience Filter): needs "audience"
- Step 5 (Generic Outline): needs "outline", "structure", "analyze"
- Step 6 (Synthetic Content): needs "write", "quiz", "abstract"
- Step 7 (Duration Limit): needs duration pattern in prompt

In [ ]:
# --- From search_service.py lines 1146-1186 ---
# All steps check for keywords in prompt.lower() - since prompt is "", ALL are False

print("Pipeline Steps 2-7 Check (prompt is empty):")
print()

# Step 2: Inclusion/Exclusion
step2_keywords = ["include", "exclude", "don't", "only", "remove", "skip", "ensure"]
should_run_step_2 = any(w in prompt.lower() for w in step2_keywords)
print(f"  Step 2 (Inclusion/Exclusion): {'RUN' if should_run_step_2 else 'SKIPPED'}")

# Step 3: Audience Filter
step3_keywords = ["audience"]
should_run_step_3 = any(w in prompt.lower() for w in step3_keywords)
print(f"  Step 3 (Audience Filter):      {'RUN' if should_run_step_3 else 'SKIPPED'}")

# Step 5: Generic Outline
step5_keywords = ["outline", "structure", "analyze"]
should_run_step_5 = not all_topics and any(w in prompt.lower() for w in step5_keywords)
print(f"  Step 5 (Generic Outline):      {'RUN' if should_run_step_5 else 'SKIPPED'}")

# Step 6: Synthetic Content
step6_keywords = ["write", "quiz", "abstract"]
should_run_step_6 = any(w in prompt.lower() for w in step6_keywords)
print(f"  Step 6 (Synthetic Content):    {'RUN' if should_run_step_6 else 'SKIPPED'}")

# Step 7: Duration Limit
def extract_duration_from_prompt(p):
    # Simplified - checks for duration patterns like "30 minutes", "1 hour"
    match = re.search(r'(\d+)\s*(?:min|minute|hour|hr)', p, re.IGNORECASE)
    return int(match.group(1)) if match else None

target_mins = extract_duration_from_prompt(prompt)
print(f"  Step 7 (Duration Limit):       {'RUN (' + str(target_mins) + ' mins)' if target_mins else 'SKIPPED'}")

print(f"\nPool size unchanged: {len(current_pool)} chunks")

## Step 12: Sequencing Logic (Steps 8-9)
**From:** `search_service.py` → `OrchestratorAgent.process_search_request()` (lines 1190-1221)

Sequencing depends on **file count**:
- **Single file** → `apply_page_number_sorting()` (sort by page number)
- **Multi-file** → `apply_llm_logical_sequencing()` (LLM reorders logically) + optional manual sequencing

In [ ]:
# --- From search_service.py lines 1190-1221 (Sequencing Logic) ---

files_list = files if isinstance(files, list) else ([files] if files else [])
is_single_file = (len(files_list) == 1)

print(f"Files: {files_list}")
print(f"Is single file: {is_single_file}")
print()

if is_single_file:
    # --- CASE A: Single File -> Page Number Sorting ---
    # From search_service.py lines 1025-1047 (apply_page_number_sorting)
    print("CASE A: Single file -> apply_page_number_sorting()")
    print("  Sorting all chunks by page number (ascending)...")
    
    def get_page_number(item):
        raw_page = str(item.get('Source Page', '999999'))
        match = re.search(r'(\d+)', raw_page)
        return int(match.group(1)) if match else 999999
    
    current_pool = sorted(current_pool, key=get_page_number)
    print(f"  Sorted {len(current_pool)} chunks by page number")
    
    # Show page order
    pages = [str(get_page_number(c)) for c in current_pool[:10]]
    print(f"  First 10 page numbers: {pages}")

else:
    # --- CASE B: Multi File -> LLM Logical Sequencing ---
    # From search_service.py lines 877-927 (apply_llm_logical_sequencing)
    print("CASE B: Multi file -> apply_llm_logical_sequencing()")
    print("  LLM will reorder chunks for best logical flow...")
    print("  (Intro -> Concepts -> Advanced -> Summary)")
    
    if len(current_pool) > 1:
        # Build lightweight input for LLM
        item_map = {i: c for i, c in enumerate(current_pool)}
        llm_input = [{
            "index": i, 
            "Topic": c.get('Topic', ''), 
            "Subtopic": c.get('Subtopic', ''),
            "Chapter": c.get('Chapter', '')
        } for i, c in enumerate(current_pool)]
        
        system_msg = """
        You are an Instructional Designer. Reorder indices for best logical flow.
        Flow: Intro -> Concepts -> Advanced -> Summary.
        Return JSON { 'ordered_indices': [0, 4, 2...] } using ALL indices.
        """
        
        try:
            res = chat_client.chat.completions.create(
                model=deployment_name,
                messages=[
                    {"role": "system", "content": system_msg},
                    {"role": "user", "content": json.dumps(llm_input)}
                ],
                response_format={"type": "json_object"}, temperature=0.0, seed=GLOBAL_SEED
            )
            payload = json.loads(res.choices[0].message.content)
            new_order_indices = payload.get("ordered_indices", [])
            
            if set(item_map.keys()) == set(new_order_indices):
                current_pool = [item_map[idx] for idx in new_order_indices]
                print(f"  Re-ordered {len(current_pool)} chunks successfully")
            else:
                print("  Indices mismatch - keeping original order")
        except Exception as e:
            print(f"  LLM sequencing failed: {e} - keeping original order")
    
    # Step 9: Manual Sequencing - SKIPPED (no ordering keywords in empty prompt)
    manual_keywords = ["organise", "first", "last", "then", "start", "end", "order", "sequence", "arrange", "organize"]
    has_manual_seq = any(w in prompt.lower() for w in manual_keywords)
    print(f"\n  Step 9 (Manual Sequencing): {'RUN' if has_manual_seq else 'SKIPPED (no ordering keywords)'}")

print(f"\nFinal pool after sequencing: {len(current_pool)} chunks")

## Step 13: Save Results (Inject Intro, Knowledge Checks, Summary, Assessment)
**From:** `search_service.py` → `OrchestratorAgent.save_results()` (lines 1244-1283)

This is where the pipeline **enhances** the raw results by injecting:
1. **3 Intro rows**: "Welcome", "Navigation Tour", "Course Overview and Objectives"
2. **Knowledge Check** rows between chapter transitions
3. **Course Summary** and **Assessment** rows at the end

In [ ]:
# --- From search_service.py lines 1244-1283 (OrchestratorAgent.save_results) ---

def save_results(results, metadata_inputs, filename_prefix, output_filename=None):
    if not results: 
        print("No results to save!")
        return
    
    # Calculate total duration
    total_seconds = sum(result.get('duration_seconds', 0) for result in results)
    total_duration = f"{int(total_seconds // 60)} min {int(total_seconds % 60)} sec"
    print(f"Total content duration: {total_duration}")
    
    # Template row (empty row with all fields)
    template_row = {key: "" for key in json_field_mapping.keys()}
    
    # --- INJECT: 3 Introduction rows at the start ---
    enhanced_results = [
        {**template_row, "Chapter": "Introductions", "Topic": "Welcome"},
        {**template_row, "Chapter": "Introductions", "Topic": "Navigation Tour"},
        {**template_row, "Chapter": "Introductions", "Topic": "Course Overview and Objectives"}
    ]
    print(f"\nInjected 3 Introduction rows")
    
    # --- INJECT: Knowledge Check rows between chapter transitions ---
    error = results[0].get("error") if results and results[0].get("error") else None
    last_chapter = None
    knowledge_checks_added = 0
    
    for row in results:
        if last_chapter and row.get("Chapter") != last_chapter:
            if "introduction" not in last_chapter.lower():
                enhanced_results.append({
                    **template_row, 
                    "Chapter": last_chapter, 
                    "Topic": "Knowledge Check", 
                    "error": error
                })
                knowledge_checks_added += 1
        enhanced_results.append(row)
        last_chapter = row.get("Chapter")
    
    # Final Knowledge Check for last chapter
    if last_chapter:
        enhanced_results.append({
            **template_row, "Chapter": last_chapter, "Topic": "Knowledge Check", "error": error
        })
        knowledge_checks_added += 1
    
    print(f"Injected {knowledge_checks_added} Knowledge Check rows")
    
    # --- INJECT: Course Summary and Assessment at the end ---
    enhanced_results.extend([
        {**template_row, "Chapter": "Course Summary", "Topic": "", "error": error},
        {**template_row, "Chapter": "Assessment", "Topic": "", "error": error}
    ])
    print(f"Injected Course Summary and Assessment rows")
    
    # --- Build final output JSON ---
    output_data = {
        "metadata": {**metadata_inputs, "Duration": total_duration, "error": error},
        "results": [{k: v for k, v in row.items() if k in json_field_mapping} for row in enhanced_results]
    }
    
    # Save to file
    filename = output_filename if output_filename else f"{filename_prefix}"
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(output_data, f, indent=4, ensure_ascii=False)
    
    print(f"\nSaved to: {filename}")
    print(f"Total rows in output: {len(output_data['results'])} (original: {len(results)}, injected: {len(output_data['results']) - len(results)})")
    return output_data

# Build metadata (from search_service.py lines 1228-1234)
base_metadata = {
    "Client": CLIENT,
    "Project": PROJECT,
    "Context Prompt": prompt,
    "Files": ", ".join(files) if isinstance(files, list) else str(files),
    "Topic": "Auto-Detected"
}

# Generate unique output filename (from orchestrator.py lines 593-595)
request_id = str(uuid.uuid4())
timestamp = int(time.time())
output_filename = f"outline_{request_id}_{timestamp}.json"

# Execute save
output_data = save_results(current_pool, base_metadata, "Orchestrator_Output", output_filename)

## Step 14: Read JSON & Map to Outline Items
**From:** `orchestrator.py` → `search_outline_pipeline()` (lines 616-642)

The orchestrator reads back the saved JSON file and maps each result to the final outline format that the frontend expects.

In [ ]:
# --- From orchestrator.py lines 616-642 (search_outline_pipeline - result mapping) ---

# Read the saved JSON file
with open(output_filename, 'r', encoding='utf-8') as f:
    data = json.load(f)

metadata = data.get("metadata")
error = metadata.get("error", "")
print("Metadata:")
print(json.dumps(metadata, indent=2, default=str))
print()

# Map results to final outline format
if 'results' in data:
    final_outline = data['results']
    mapped_outline = []
    for item in final_outline:
        mapped_item = {
            "File": item.get("File", ""),
            "Source Page": item.get("Source Page", ""),
            "Chapter": item.get("Chapter", ""),
            "Topic": item.get("Topic", ""),
            "Subtopic": item.get("Subtopic", ""),
            "Full Page Content": item.get("Full Page Content", ""),
            "Durations (Mins)": item.get("Durations (Mins)", ""),
            "error": error
        }
        mapped_outline.append(mapped_item)
    
    print(f"Mapped outline items: {len(mapped_outline)}")
else:
    mapped_outline = []
    print("WARNING: No 'results' key found in output file!")

## Step 15: Inspect Final Outline (What Frontend Receives)
Display the complete outline as a table - this is exactly what the frontend renders.

In [ ]:
# Display the final outline as a table
df_outline = pd.DataFrame(mapped_outline)

# Show summary columns (exclude heavy content)
display_cols = ["File", "Source Page", "Chapter", "Topic", "Subtopic", "Durations (Mins)"]
available_cols = [c for c in display_cols if c in df_outline.columns]

print(f"Total outline rows: {len(df_outline)}")
print(f"Columns: {list(df_outline.columns)}")
print()

# Display the outline table
df_outline[available_cols]

## Step 16: Inspect Injected Rows (Intro, Knowledge Checks, Summary)
See which rows were auto-injected vs which came from actual content.

In [ ]:
# Categorize rows: Injected vs Content
def categorize_row(row):
    chapter = row.get("Chapter", "")
    topic = row.get("Topic", "")
    file_val = row.get("File", "")
    
    if chapter == "Introductions":
        return "INJECTED - Introduction"
    elif topic == "Knowledge Check":
        return "INJECTED - Knowledge Check"
    elif chapter == "Course Summary":
        return "INJECTED - Course Summary"
    elif chapter == "Assessment":
        return "INJECTED - Assessment"
    elif file_val:
        return "CONTENT - From Source File"
    else:
        return "UNKNOWN"

df_outline["Row Type"] = df_outline.apply(categorize_row, axis=1)

print("Row Type Breakdown:")
print(df_outline["Row Type"].value_counts().to_string())
print()
print("Chapter Distribution:")
print(df_outline["Chapter"].value_counts().to_string())

## Step 17: Simulate Error Checking (run_outline_job)
**From:** `outline_routes.py` → `run_outline_job()` (lines 206-248)

After the outline is generated, the background job checks for errors before setting the result.

In [ ]:
# --- From outline_routes.py lines 206-248 (run_outline_job error checking) ---

outline = mapped_outline  # This is what OutlineService.generate_outline() returns

# Check 1: Empty outline
if not outline:
    print("ERROR: Empty outline - file processed incorrectly")
else:
    print(f"Outline is not empty ({len(outline)} items)")

# Check 2: Error in first item
if outline and len(outline) > 0:
    first_item = outline[0]
    error_message = None
    
    if "error" in first_item and first_item["error"]:
        error_message = first_item["error"]
    elif first_item.get("Topic") == "Error":
        error_message = first_item.get("content") or first_item.get("Content_without_images", "Error occurred")
    elif "No search results found" in str(first_item):
        error_message = str(first_item)
    
    if error_message:
        print(f"ERROR detected: {error_message}")
    else:
        print("No errors detected - outline generation SUCCESSFUL")
        print(f"\nFinal response to frontend:")
        print(f'  {{"status": "done", "outline": [{len(outline)} items...]}}')

## Step 18: Cleanup
Remove the temporary output JSON file created during testing.

In [ ]:
# Cleanup: Remove temporary output file
if os.path.exists(output_filename):
    os.remove(output_filename)
    print(f"Cleaned up: {output_filename}")
else:
    print("No file to clean up")